# 1. IMPORTACIÓN DE DATOS

In [1]:
# Parámetros a definir

    # lesk: True/False
        # True --> Synset se elige utilizando el desambiguador de Lesk
        # False --> Synset es el más común 
    # tipo_bow: --> Categorías de synsets que cogeremos para hacer los cálculos
        # todos (1) --> Todos los synsets
        # nv (2) --> Solo nombres y adjetivos
    # metodo_frase: lw / Ariadna / Agirre
    # partición: numero entero
        # fragmento del corpus que se procesa.
        

desambiguar = True

expandir_contracciones = False

tipo_bow = 'nv'

metodo_frase = 'lw'

particion = 1

print_info = False

importar_librerias = True

importar_datos = True

criterios = 'Lesk-' + str(desambiguar) + ', tipo_bow-' + str(tipo_bow) + ', metodo-' + str(metodo_frase) + ', particion-' + str(particion)

print()
print('Desambiguación desambiguar:', desambiguar)
print('Tipo de BoW:', tipo_bow)
print('Método de comparación a nivel de frase:', metodo_frase)
print('Particion de filas a importar:', particion)
print('Importar librerías:', importar_librerias)
print('Importar datos:', importar_datos)

print(criterios)




Desambiguación desambiguar: True
Tipo de BoW: nv
Método de comparación a nivel de frase: lw
Particion de filas a importar: 1
Importar librerías: True
Importar datos: True
Lesk-True, tipo_bow-nv, metodo-lw, particion-1


In [2]:
# Importación de librerías

if importar_librerias:
    
    import time
    import datetime

    import csv

    import pandas as pd
    pd.options.display.max_rows  # Para mostrar todas las columnas
    pd.set_option('display.max_colwidth', -1)  # Para incluir todo el contenido de una celda, sin truncar contenido.
    pd.set_option('display.max_columns', 500)  # Para incluir todas las columnas (no serán más de 500)
    pd.set_option('display.max_rows', 6000)  # Para incluir todas las columnas (no serán más de 6000)

    import numpy as np

    from nltk.corpus import wordnet as wn

    from nltk.corpus import wordnet_ic
    ic_brown = wordnet_ic.ic('ic-brown.dat')
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    
    if desambiguar == True:
        from nltk.wsd import lesk



In [3]:
# Funciones generales

# función coseno
    # Entrada: dos vectores
    # Salida: Un número real entre [0,1] (coseno entre los dos vectores)

def coseno(v1, v2):
    
    coseno = v1.dot(v2) / np.sqrt(v1.dot(v1) * v2.dot(v2))
    
    return(coseno)

# Ejemplo para comprobar funcionamiento coseno()

print('****** EJEMPLO coseno() ********')   
v1 = np.array([1,2,3])
v2 = np.array([1,0,1])
print('Coseno:', coseno(v1,v1))
print('Coseno:', coseno(v1,v2))
print('****')


****** EJEMPLO coseno() ********
Coseno: 1.0
Coseno: 0.7559289460184544
****


In [4]:
# Importación de los datos del fichero sts-train.csv a un dataframe

start_total = time.time()

if importar_datos:

    file = 'stsbenchmark\sts-train.csv'
    corpus = pd.read_csv(file, sep='\t', error_bad_lines=False, quoting=csv.QUOTE_NONE, usecols=range(7), header=-1)
    corpus.head()
    corpus.columns = ['genre', 'file', 'year', 'id', 'gold_score', 'sent1', 'sent2']


end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

corpus.sample(5)

TIEMPO TRANSCURRIDO: 0.037199974060058594


,genre,file,year,id,gold_score,sent1,sent2
1309,main-captions,images,2014,450,3.8,Black and white cow standing in grassy field.,A black and white horned cow standing in a field.
3710,main-news,deft-news,2014,260,2.0,north korean officials are suspected of building nuclear weapons.,north korea is currently in the process of dismantling its nuclear weapons program.
5206,main-news,headlines,2015,638,0.4,More rain hampers Indian flood rescue,US tourist gang-raped in northern India: Police
569,main-captions,MSRvid,2012train,110,2.0,A person is slicing a scallop.,A person is slicing octopus.
1002,main-captions,images,2014,3,3.2,A large boat in the water at the marina.,A large boat on the sea.


# 2. ANÁLISIS 1: REVISIÓN DE LOS DATOS BRUTOS (...)

In [5]:
# Eliminación de duplicados

if importar_datos:

    corpus.drop_duplicates(['sent1', 'sent2'], keep='first', inplace=True)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

TIEMPO TRANSCURRIDO: 0.0793619155883789


In [6]:
# Creación de un corpus c reducido con filas representativas del corpus original

    # Para realizar pre-procesamientos que nos permitan verificar la adecuación de los algoritmos
    # en un tiempo reducido.

particion = particion

c = corpus.iloc[::particion, :].copy()
num_filas = 'Número de filas incluídas en el dataframe: ' + str(c.shape[0])

print(num_filas)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head(12)

Número de filas incluídas en el dataframe: 5706
TIEMPO TRANSCURRIDO: 0.08878397941589355


,genre,file,year,id,gold_score,sent1,sent2
0,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.
1,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.
2,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncooked pizza.
3,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.
4,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.
5,main-captions,MSRvid,2012test,11,4.25,Some men are fighting.,Two men are fighting.
6,main-captions,MSRvid,2012test,12,0.50,A man is smoking.,A man is skating.
7,main-captions,MSRvid,2012test,13,1.60,The man is playing the piano.,The man is playing the guitar.
8,main-captions,MSRvid,2012test,14,2.20,A man is playing on a guitar and singing.,A woman is playing an acoustic guitar and singing.
9,main-captions,MSRvid,2012test,16,5.00,A person is throwing a cat on to the ceiling.,A person throws a cat on the ceiling.


# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN

In [7]:
# Pre procesamiento
# https://www.linkedin.com/pulse/processing-normalizing-text-data-saurav-ghosh/
# http://xperimentallearning.blogspot.com/2018/05/nlppython.html

# ========================================
# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN
# ========================================

import nltk

def expandir_contracciones(texto):
    # specific
    texto = re.sub(r"won\'t", "will not", texto)
    texto = re.sub(r"can\'t", "can not", texto)

    # general
    texto = re.sub(r"n\'t", " not", texto)
    texto = re.sub(r"\'re", " are", texto)
    texto = re.sub(r"\'s", " is", texto)
    texto = re.sub(r"\'d", " would", texto)
    texto = re.sub(r"\'ll", " will", texto)
    texto = re.sub(r"\'t", " not", texto)
    texto = re.sub(r"\'ve", " have", texto)
    texto = re.sub(r"\'m", " am", texto)
    return texto

def tokenizar(sent, print_tokenizar =False):
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
    if print_tokenizar == True: print('Frase a tokenizar:', sent)
        
    pattern = r'''(?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    '''
    if print_tokenizar == True: print('pattern:', pattern)
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(sent, pattern, gaps=False)
    
    # limpieza 1: mayúsculas
    
    tokens = [token.lower() for token in tokens]   
    if print_tokenizar == True: print('Tokens:', tokens)
 
    # anotación POS. Output: lista de tuplas
    
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    
    # Convertimos las etiquetas gramaticales en notación Lesk;
        
    for i, tagged_token in enumerate(tagged_tokens):
        if tagged_tokens[i][1] == 'NOUN': tagged_tokens[i] = (tagged_tokens[i][0], 'n')
        if tagged_tokens[i][1] == 'VERB': tagged_tokens[i] = (tagged_tokens[i][0], 'v')
        if tagged_tokens[i][1] == 'ADJ': tagged_tokens[i] = (tagged_tokens[i][0], 'a')
        if tagged_tokens[i][1] == 'ADV': tagged_tokens[i] = (tagged_tokens[i][0], 'r')
    if print_tokenizar == True: print('Tagged_tokens:', tagged_tokens)
    
    # limipieza 2: stopwords - REVISAR si queremos/podemos incluir phrasal verbs, determinantes, pronombres...
            
                # Wordnet no incluye determinantes o preposiciones, de forma que "a" es asociado por Lesk 
                # al Synset('deoxyadenosine_monophosphate.n.01'), lo cual no nos conviene.
                # Por otro lado, Lesk utiliza para desambiguar solo las palabras que aparecen en Wordnet,
                # de modo que limpiar antes o después de Lesk las stopwords no debería afectar al resultado.
   
    if print_tokenizar == True: print('Limpieza de stopwords')
    stopwords = nltk.corpus.stopwords.words('english')
    tagged_stops = [tagged_stop for tagged_stop in tagged_tokens if tagged_stop[0] in stopwords]
    if print_tokenizar == True: print('Stopwords eliminadas:', tagged_stops)
    tokens = [token for token in tokens if token not in stopwords]
    tagged_tokens = [tagged_token for tagged_token in tagged_tokens if tagged_token[0] not in stopwords]
    if print_tokenizar == True: print('Tagged tokens sin stopwords:', tagged_tokens)

    return(tokens, tagged_tokens, tagged_stops)

# Ejemplo para comprobar funcionamiento tokenizar()
   
sent = "I'll always love my sweet baby"

print('****** EJEMPLO tokenizar() ********')
tokens = tokenizar(sent, print_tokenizar=True)
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)


****** EJEMPLO tokenizar() ********
Frase a tokenizar: I'll always love my sweet baby
pattern: (?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    
Tokens: ['i', 'll', 'always', 'love', 'my', 'sweet', 'baby']
Tagged_tokens: [('i', 'n'), ('ll', 'v'), ('always', 'r'), ('love', 'v'), ('my', 'PRON'), ('sweet', 'a'), ('baby', 'n')]
Limpieza de stopwords
Stopwords eliminadas: [('i', 'n'), ('ll', 'v'), ('my', 'PRON')]
Tagged tokens sin stopwords: [('always', 'r'), ('love', 'v'), ('sweet', 'a'), ('baby', 'n')]
****
TIEMPO TRANSCURRIDO: 0.2549440860748291


In [8]:
c.head()

,genre,file,year,id,gold_score,sent1,sent2
0,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.
1,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.
2,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncooked pizza.
3,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.
4,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.


In [9]:
# Tokenización de c

c['tokens1'], c['tokens2'] = '', ''
c['tagged_tokens1'], c['tagged_tokens2'] = '', ''
c['stops1'], c['stops2'] = '', ''

print_tokenizar = False

for i in c.index:
    c.at[i, 'tokens1'] = tokenizar(c.at[i, 'sent1'], print_tokenizar)[0]
    c.at[i, 'tokens2'] = tokenizar(c.at[i, 'sent2'], print_tokenizar)[0]
    c.at[i, 'tagged_tokens1'] = tokenizar(c.at[i, 'sent1'], print_tokenizar)[1]
    c.at[i, 'tagged_tokens2'] = tokenizar(c.at[i, 'sent2'], print_tokenizar)[1]
    c.at[i, 'stops1'] = tokenizar(c.at[i, 'sent1'], print_tokenizar)[2]
    c.at[i, 'stops2'] = tokenizar(c.at[i, 'sent2'], print_tokenizar)[2]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head()

TIEMPO TRANSCURRIDO: 49.33565402030945


,genre,file,year,id,gold_score,sent1,sent2,tokens1,tokens2,tagged_tokens1,tagged_tokens2,stops1,stops2
0,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.,"[plane, taking]","[air, plane, taking]","[(plane, n), (taking, v)]","[(air, n), (plane, n), (taking, v)]","[(a, DET), (is, v), (off, PRT)]","[(an, DET), (is, v), (off, PRT)]"
1,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.,"[man, playing, large, flute]","[man, playing, flute]","[(man, n), (playing, v), (large, a), (flute, n)]","[(man, n), (playing, v), (flute, n)]","[(a, DET), (is, v), (a, DET)]","[(a, DET), (is, v), (a, DET)]"
2,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncooked pizza.,"[man, spreading, shreded, cheese, pizza]","[man, spreading, shredded, cheese, uncooked, pizza]","[(man, n), (spreading, v), (shreded, v), (cheese, n), (pizza, n)]","[(man, n), (spreading, v), (shredded, v), (cheese, n), (uncooked, a), (pizza, n)]","[(a, DET), (is, v), (on, ADP), (a, DET)]","[(a, DET), (is, v), (on, ADP), (an, DET)]"
3,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.,"[three, men, playing, chess]","[two, men, playing, chess]","[(three, NUM), (men, n), (playing, v), (chess, n)]","[(two, NUM), (men, n), (playing, v), (chess, n)]","[(are, v)]","[(are, v)]"
4,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.,"[man, playing, cello]","[man, seated, playing, cello]","[(man, n), (playing, v), (cello, n)]","[(man, n), (seated, v), (playing, v), (cello, n)]","[(a, DET), (is, v), (the, DET)]","[(a, DET), (is, v), (the, DET)]"


In [10]:
# ========================================
# 4. ASIGNACIÓN DE SYNSETS DE WORDNET
# ========================================

# Asignación de synsets 
    # Entrada: lista de tuplas con tokens y sus correspondientes etiquetas gramaticales
    # Argumentos: 
         # lesk: Opción de aplicar el algoritmo de Lesk  [Lesk 1986] de la librería wsd para desambiguar
                # Opción por defecto: n (no) --> Se asignará el primer synset del grupo de synsets (el más común)
                # Opción: s (sí) --> Asignar el synset determinado por Lesk
                
def syns(tagged_tokens, desambiguar=desambiguar):
    from nltk.wsd import lesk
    for i in c.index:
        syns = list()
        errors = list()
        if desambiguar == True:
            for j in tagged_tokens:
                if lesk([token for token, tag in tagged_tokens], j[0], j[1]) is not None:
                    syns.append(lesk([token for token, tag in tagged_tokens], j[0], j[1]))
                else:
                    errors.append(j)
        else: 
            for j in tagged_tokens:
                try: syns.append(wn.synsets(j[0],j[1])[0])
                except: errors.append(j)
        return(syns, errors)
    
# Ejemplo para comprobar funcionamiento syns()

desambiguar = True

print('Desambiguación Lesk:', desambiguar)

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

desambiguar = False

print('Desambiguación Lesk:', desambiguar)

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

Desambiguación Lesk: True
TIEMPO TRANSCURRIDO: 49.38971710205078
****** EJEMPLO syns() ********
tagged_tokens: [('man', 'n'), ('cutting', 'v'), ('fish', 'a')]
syns(tagged_tokens): [Synset('world.n.08'), Synset('cut.v.24')]
syns(tagged_tokens) - Errors: [('fish', 'a')]
****
Desambiguación Lesk: False
TIEMPO TRANSCURRIDO: 49.39117670059204
****** EJEMPLO syns() ********
tagged_tokens: [('man', 'n'), ('cutting', 'v'), ('fish', 'a')]
syns(tagged_tokens): [Synset('man.n.01'), Synset('cut.v.01')]
syns(tagged_tokens) - Errors: [('fish', 'a')]
****


In [11]:
# Asignacion de synsets a los tokens de c

desambiguar = desambiguar
print('Desambiguación Lesk:', desambiguar)
tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)

c['syns1'] = ''
c['syns2'] = ''
c['errors1'] = ''
c['errors2'] = ''
c['syns1_nv'] = ''
c['syns2_nv'] = ''
c['syns1_resto'] = ''
c['syns2_resto'] = ''

for i in c.index:
    c.at[i,'syns1'] = syns(c.at[i, 'tagged_tokens1'])[0]
    c.at[i,'syns2'] = syns(c.at[i, 'tagged_tokens2'])[0]
    c.at[i,'errors1'] = syns(c.at[i, 'tagged_tokens1'])[1]
    c.at[i,'errors2'] = syns(c.at[i, 'tagged_tokens2'])[1]
    
    if tipo_bow == 'nv':
        c.at[i,'syns1_nv'] = [syn for syn in c.at[i,'syns1'] if syn.pos() in ['n', 'v']]
        c.at[i,'syns2_nv'] = [syn for syn in c.at[i,'syns2'] if syn.pos() in ['n', 'v']]
        c.at[i,'syns1_resto'] = [syn for syn in c.at[i,'syns1'] if syn.pos() not in ['n', 'v']]
        c.at[i,'syns2_resto'] = [syn for syn in c.at[i,'syns2'] if syn.pos() not in ['n', 'v']]


end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

Desambiguación Lesk: False
Tipo_bow: nv
TIEMPO TRANSCURRIDO: 65.75818014144897


,genre,file,year,id,gold_score,sent1,sent2,tokens1,tokens2,tagged_tokens1,tagged_tokens2,stops1,stops2,syns1,syns2,errors1,errors2,syns1_nv,syns2_nv,syns1_resto,syns2_resto
1264,main-captions,images,2014,387,2.2,Red double decker bus taking on passengers.,A red double decker bus driving down a street.,"[red, double, decker, bus, taking, passengers]","[red, double, decker, bus, driving, street]","[(red, a), (double, a), (decker, n), (bus, n), (taking, v), (passengers, n)]","[(red, a), (double, a), (decker, n), (bus, n), (driving, v), (street, n)]","[(on, ADP)]","[(a, DET), (down, PRT), (a, DET)]","[Synset('double.a.04'), Synset('dekker.n.01'), Synset('busbar.n.01'), Synset('remove.v.01'), Synset('passenger.n.01')]","[Synset('double.a.04'), Synset('dekker.n.01'), Synset('busbar.n.01'), Synset('drive.v.03'), Synset('street.n.05')]","[(red, a)]","[(red, a)]","[Synset('dekker.n.01'), Synset('busbar.n.01'), Synset('remove.v.01'), Synset('passenger.n.01')]","[Synset('dekker.n.01'), Synset('busbar.n.01'), Synset('drive.v.03'), Synset('street.n.05')]",[Synset('double.a.04')],[Synset('double.a.04')]
1093,main-captions,images,2014,130,3.8,Two cats sitting together on the sofa looking out the window.,Two cats are looking at a window.,"[two, cats, sitting, together, sofa, looking, window]","[two, cats, looking, window]","[(two, NUM), (cats, n), (sitting, v), (together, r), (sofa, n), (looking, v), (window, n)]","[(two, NUM), (cats, n), (looking, v), (window, n)]","[(on, ADP), (the, DET), (out, PRT), (the, DET)]","[(are, v), (at, ADP), (a, DET)]","[Synset('caterpillar.n.02'), Synset('sit_down.v.01'), Synset('together.r.05'), Synset('sofa.n.01'), Synset('search.v.02'), Synset('windowpane.n.01')]","[Synset('caterpillar.n.02'), Synset('search.v.02'), Synset('windowpane.n.01')]","[(two, NUM)]","[(two, NUM)]","[Synset('caterpillar.n.02'), Synset('sit_down.v.01'), Synset('sofa.n.01'), Synset('search.v.02'), Synset('windowpane.n.01')]","[Synset('caterpillar.n.02'), Synset('search.v.02'), Synset('windowpane.n.01')]",[Synset('together.r.05')],[]
4559,main-news,headlines,2014,264,2.6,George Zimmerman found not guilty of Trayvon Martin murder,Protests and appeals for calm after Zimmerman acquitted of Trayvon Martin murder,"[george, zimmerman, found, guilty, trayvon, martin, murder]","[protests, appeals, calm, zimmerman, acquitted, trayvon, martin, murder]","[(george, n), (zimmerman, n), (found, v), (guilty, a), (trayvon, n), (martin, n), (murder, n)]","[(protests, n), (appeals, n), (calm, n), (zimmerman, n), (acquitted, v), (trayvon, n), (martin, n), (murder, n)]","[(not, r), (of, ADP)]","[(and, CONJ), (for, ADP), (after, ADP), (of, ADP)]","[Synset('george.n.07'), Synset('establish.v.08'), Synset('guilty.a.01'), Synset('martin.n.05'), Synset('murder.n.01')]","[Synset('protest.n.03'), Synset('solicitation.n.02'), Synset('composure.n.01'), Synset('behave.v.02'), Synset('martin.n.05'), Synset('murder.n.01')]","[(zimmerman, n), (trayvon, n)]","[(zimmerman, n), (trayvon, n)]","[Synset('george.n.07'), Synset('establish.v.08'), Synset('martin.n.05'), Synset('murder.n.01')]","[Synset('protest.n.03'), Synset('solicitation.n.02'), Synset('composure.n.01'), Synset('behave.v.02'), Synset('martin.n.05'), Synset('murder.n.01')]",[Synset('guilty.a.01')],[]
429,main-captions,MSRvid,2012test,642,2.8,The woman is pencilling on eye shadow.,The girl is using an eye pencil on her eyelid.,"[woman, pencilling, eye, shadow]","[girl, using, eye, pencil, eyelid]","[(woman, n), (pencilling, v), (eye, n), (shadow, n)]","[(girl, n), (using, v), (eye, n), (pencil, n), (eyelid, n)]","[(the, DET), (is, v), (on, ADP)]","[(the, DET), (is, v), (an, DET), (on, ADP), (her, PRON)]","[Synset('womanhood.n.02'), Synset('pencil.v.01'), Synset('eye.n.05'), Synset('trace.n.02')]","[Synset('girlfriend.n.02'), Synset('use.v.04'), Synset('eye.n.05'), Synset('pencil.n.04'), Synset('eyelid.n.01')]",[],[],"[Synset('womanhood.n.02'), Synset('pencil.v.01'), Synset('eye.n.05'), Synset('trace.n.02')]","[

# 4. ANÁLISIS 2: TIPO Y DISTRIBUCIÓN DE TOKENS Y SYNSETS (...)
# 5. BAG OF WORDS (BoW)

In [12]:
# ========================================
# 5. BAG OF WORDS
# ========================================

# Creación de la Bag of Words (BoW) (de hecho, será una "Bag of synsets")
    # La necesitaremos luego como espacio vectorial para aplicar el método [Liu 2013] de similitud a nivel de frase.

tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)
    
c['bow'] = c['syns1'] + c['syns2']
c['bow_errors'] =  c['errors1'] + c['errors2']
c['bow_nv'] =  c['syns1_nv'] + c['syns2_nv']
c['bow_resto'] =  c['syns1_resto'] + c['syns2_resto']

for i in c.index:
    c.at[i, 'bow'] = list(set(c.at[i, 'bow']))
    c.at[i, 'bow_errors'] = list(set(c.at[i, 'bow_errors']))    
    if tipo_bow == 'nv':
        c.at[i, 'bow_nv'] = list(set(c.at[i, 'bow_nv']))
        c.at[i, 'bow_resto'] = list(set(c.at[i, 'bow_resto']))

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

Tipo_bow: nv
TIEMPO TRANSCURRIDO: 66.24676990509033


,genre,file,year,id,gold_score,sent1,sent2,tokens1,tokens2,tagged_tokens1,tagged_tokens2,stops1,stops2,syns1,syns2,errors1,errors2,syns1_nv,syns2_nv,syns1_resto,syns2_resto,bow,bow_errors,bow_nv,bow_resto
3661,main-news,deft-news,2014,211,5.0,putin stated that the russian government could withdraw from the treaty altogether if western nations refuse to ratify the amended treaty.,putin stated that russia could discard the treaty entirely if western nations refuse to ratify its amended version.,"[putin, stated, russian, government, could, withdraw, treaty, altogether, western, nations, refuse, ratify, amended, treaty]","[putin, stated, russia, could, discard, treaty, entirely, western, nations, refuse, ratify, amended, version]","[(putin, n), (stated, v), (russian, a), (government, n), (could, v), (withdraw, v), (treaty, n), (altogether, r), (western, a), (nations, n), (refuse, v), (ratify, v), (amended, v), (treaty, n)]","[(putin, n), (stated, v), (russia, n), (could, v), (discard, v), (treaty, n), (entirely, r), (western, a), (nations, n), (refuse, v), (ratify, v), (amended, a), (version, n)]","[(that, ADP), (the, DET), (from, ADP), (the, DET), (if, ADP), (to, PRT), (the, DET)]","[(that, ADP), (the, DET), (if, ADP), (to, PRT), (its, PRON)]","[Synset('putin.n.01'), Synset('submit.v.02'), Synset('russian.a.01'), Synset('politics.n.02'), Synset('retire.v.02'), Synset('treaty.n.01'), Synset('wholly.r.01'), Synset('western.a.01'), Synset('state.n.04'), Synset('reject.v.06'), Synset('sign.v.02'), Synset('rectify.v.04'), Synset('treaty.n.01')]","[Synset('putin.n.01'), Synset('submit.v.02'), Synset('soviet_union.n.01'), Synset('discard.v.01'), Synset('treaty.n.01'), Synset('wholly.r.01'), Synset('western.a.01'), Synset('state.n.04'), Synset('reject.v.06'), Synset('sign.v.02'), Synset('amended.a.01'), Synset('version.n.06')]","[(could, v)]","[(could, v)]","[Synset('putin.n.01'), Synset('submit.v.02'), Synset('politics.n.02'), Synset('retire.v.02'), Synset('treaty.n.01'), Synset('state.n.04'), Synset('reject.v.06'), Synset('sign.v.02'), Synset('rectify.v.04'), Synset('treaty.n.01')]","[Synset('putin.n.01'), Synset('submit.v.02'), Synset('soviet_union.n.01'), Synset('discard.v.01'), Synset('treaty.n.01'), Synset('state.n.04'), Synset('reject.v.06'), Synset('sign.v.02'), Synset('version.n.06')]","[Synset('russian.a.01'), Synset('wholly.r.01'), Synset('western.a.01')]","[Synset('wholly.r.01'), Synset('western.a.01'), Synset('amended.a.01')]","[Synset('western.a.01'), Synset('soviet_union.n.01'), Synset('russian.a.01'), Synset('sign.v.02'), Synset('version.n.06'), Synset('state.n.04'), Synset('retire.v.02'), Synset('politics.n.02'), Synset('treaty.n.01'), Synset('rectify.v.04'), Synset('reject.v.06'), Synset('wholly.r.01'), Synset('submit.v.02'), Synset('putin.n.01'), Synset('amended.a.01'), Synset('discard.v.01')]","[(could, v)]","[Synset('soviet_union.n.01'), Synset('sign.v.02'), Synset('version.n.06'), Synset('state.n.04'), Synset('retire.v.02'), Synset('politics.n.02'), Synset('treaty.n.01'), Synset('rectify.v.04'), Synset('reject.v.06'), Synset('submit.v.02'), Synset('putin.n.01'), Synset('discard.v.01')]","[Synset('western.a.01'), Synset('wholly.r.01'), Synset('russian.a.01'), Synset('amended.a.01')]"
278,main-captions,MSRvid,2012test,414,3.4,A man is swinging from a rope attached to the ceiling.,A man is swinging on a rope.,"[man, swinging, rope, attached, ceiling]","[man, swinging, rope]","[(man, n), (swinging, v), (rope, n), (attached, v), (ceiling, n)]","[(man, n), (swinging, v), (rope, n)]","[(a, DET), (is, v), (from, ADP), (a, DET), (to, PRT), (the, DET)]","[(a, DET), (is, v), (on, ADP), (a, DET)]","[Synset('world.n.08'), Synset('swing.v.03'), Synset('rope.n.01'), Synset('attach.v.03'), Synset('ceiling.n.04')]","[Synset('world.n.08'), Synset('swing.v.03'), Synset('rope.n.01')]",[],[],"[Synset('world.n.08'), Synset('swing.v.03'), Synset('rope.n.01'), Synset('attach.v.03'), Synset('ceiling.n.04')]","[Synset('world.n.08'), Synset('

# 6. SIMILITUD A NIVEL LÉXICO - MÉTODOS INCLUÍDOS EN WORDNET

In [13]:
# ========================================
# 6. SIMILITUD A NIVEL LÉXICO - MÉTODOS INCLUÍDOS EN WORDNET
# ========================================

# Ejemplos de funciones de similitud integradas en Wordnet

    # ps --> path_similarity (POR DEFECTO). Rango: [0:1]
    # lch --> Leacock Chodorow. Rango: [0:3.64?] 
    # wup --> Wu_Palmer
    # res --> Resknik (requiere corpus ic)
    # jcn --> Jiang-Conrath (requiere corpus ic)
    # lin --> Lin (requiere corpus ic)

# Ejemplos para comprobar funcionamiento de los distintos métodos de simlitud de wordnet
    
syn1 = wn.synset('dog.n.01')
syn2 = wn.synset('cat.n.01')
syn3 = wn.synset('black.a.01')
syn4 = wn.synset('white.a.01')

try: print('wn.path_similarity(', syn1, syn1, '):',  wn.path_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn1, syn2, '):',  wn.path_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn1, syn3, '):',  wn.path_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn3, syn4, '):',  wn.path_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)
    
print()

try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.lch_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.lch_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: rint('wn.lch_similarity(', syn1, syn3, '):',  wn.lch_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.lch_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)    

print()

try: print('wn.wup_similarity(', syn1, syn1, '):',  wn.wup_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn1, syn2, '):',  wn.wup_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn1, syn3, '):',  wn.wup_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn3, syn4, '):',  wn.wup_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)    

print()

ic = ic_brown
print('ic_brown')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

ic = ic_semcor
print('ic_brown')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

ic = ic_genesis
print('ic_brown')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

wn.path_similarity( Synset('dog.n.01') Synset('dog.n.01') ): 1.0
wn.path_similarity( Synset('dog.n.01') Synset('cat.n.01') ): 0.2
wn.path_similarity( Synset('dog.n.01') Synset('black.a.01') ): None
wn.path_similarity( Synset('black.a.01') Synset('white.a.01') ): None

wn.lch_similarity( Synset('dog.n.01') Synset('dog.n.01') ): 3.6375861597263857
wn.lch_similarity( Synset('dog.n.01') Synset('cat.n.01') ): 2.0281482472922856
ERROR name 'rint' is not defined
wn.lch_similarity( Synset('black.a.01') Synset('white.a.01') ): None

wn.wup_similarity( Synset('dog.n.01') Synset('dog.n.01') ): 0.9285714285714286
wn.wup_similarity( Synset('dog.n.01') Synset('cat.n.01') ): 0.8571428571428571
wn.wup_similarity( Synset('dog.n.01') Synset('black.a.01') ): None
wn.wup_similarity( Synset('black.a.01') Synset('white.a.01') ): None

ic_brown
wn.lch_similarity( Synset('dog.n.01') Synset('dog.n.01') ): 9.006014398918229
wn.lch_similarity( Synset('dog.n.01') Synset('cat.n.01') ): 7.911666509036577
ERROR Comp

# 7. SIMILITUD A NIVEL DE FRASE - LIU-WANG


In [14]:
# ========================================
# 8. SIMILITUD A NIVEL DE FRASE - LIU-WANG
# ========================================

# 8.1 Vector semántico
    # Entrada: Un conjunto de synsets (frase) y una bag of words (bow) conteniendo el conjunto de synsets
        # si el parámetro "info" es "print", se mostrará información durante el proceso de cálculo
    # Argumentos: 
        # medida : función de similitud léxica que se quiere usar:
            # ps --> path_similarity (POR DEFECTO). 
                # Rango: [0:1]
            # lch --> Leacock Chodorow. 
                # Rango: [0:3.64?] 
            # wup --> Wu_Palmer
                # Rango: ]
            # res --> Resknik (requiere corpus ic)
                # Rango = [0-14,47]
                # Mismo synset --> Depende del synset (IC)
                # La palabra no está en el corpus --> infinito
                    # MODIFICACIÓN: Asignamos la media ponderada dentro del rango --> 9,04
            # jcn --> Jiang-Conrath (requiere corpus ic)
                # Rango = [0,1]
                # Mismo synset --> similitud = infinito (e+300)
                    # MODIFICACIÓN: Asignamos mismo synset --> similitud = 1
            # lin --> Lin (requiere corpus ic)
            # lw --> Liu-Wang
        # ic: corpus ic, para las funciones que lo requieran.
            # Intentar reproducir https://www.kdnuggets.com/2017/11/building-wikipedia-text-corpus-nlp.html
        # info: si 'print', se imprimirá información sobre el procesamiento.
    # Requerimientos: Haber importado wordnet_ic (si se utilizan las opciones res,jcn,lin)
    # Salida: Vector semántico del conjunto de synsets en el espacio definido por bow

def v_semantico(syns, bow, medida='ps', ic = '', info=''):
    v = np.zeros(len(bow), dtype=np.float)    

    if info == 'print': 
        print('bow:', bow)
        print('syns:', syns)
        print()
        print('Cálculo del vector semántico v correspondiente a syns en el espacio bow')
        print('v:', v)
        print()    
        print('Método de similitud léxica:', medida)
        print()       

    for index, synset in enumerate(bow):  
        if info == 'print': 
            print('Componente de v (index):', index)
            print('synset de bow correspondiente a index =',index, ':', synset)
            print('Valor inicial de la componente v', index, ':', v[index])
            print()
            print('Calculo del valor de la componente v', index)
            print()
        sims = list()
        for syn in syns:
            if info == 'print':
                print('Cálculo de la similitud de synset de bow con synset de syns:', synset, '<-->', syn)
                print()
            sim = 0.0
            if syn.pos() == synset.pos():
                if medida == 'ps':
                    sim = wn.path_similarity(syn, synset)
                elif medida == 'lch':
                    sim = wn.lch_similarity(syn, synset)
                elif medida == 'wup':
                    sim = wn.wup_similarity(syn, synset)
                elif medida == 'res':                    
                    sim = wn.res_similarity(syn, synset, ic)
                    if sim == 1.00000000e+300:
                        sim = 9.04
                elif medida == 'jcn':
                    if syn == synset:
                        sim = 1
                    else:
                        sim = wn.jcn_similarity(syn, synset, ic)
                elif medida == 'lin':
                    sim = wn.lin_similarity(syn, synset, ic)
                if info == 'print':
                    print('similitud:', sim)
                if sim is None: continue
                    
            else:
                sim = 0.0
                if info == 'print':
                    print(synset, 'y', syn, 'no pertenecen a la misma categoría gramatical')    
                    print('Similitud Liu ', synset, '<-->', syn, ':', sim)
            if sim != 'nan': 
                sims.append(sim)
            if info == 'print': print()
        v[index] = max(sims)
        if info == 'print':
            print('Valores de las similitudes:', sims)
            print("Valor de la componente v", index, '(máximo de las similitudes):', v[index])
            print()
    if info == 'print': print('v = ', v)
    if info == 'print': print()
    return(v)

    
# Ejemplo para comprobar funcionamiento v_semantico()

print('****** EJEMPLO v_semantico() ********')   

bow = c['bow'][501]
syns1 = c['syns1'][501]
syns2 = c['syns2'][501]
info = ''
ic = ''

print('bow:', bow)
print('syns1:', syns1)
print('syns2:', syns2)

print()
print('SIMILITUD LIU - WANG')

medida = 'lw'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE NO REQUIEREN IC')

medida = 'ps'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lch'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'wup'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = BROWN')

ic = ic_brown

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow,medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = SEMCOR')
     
ic = ic_semcor

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = GENESIS')
     
ic = ic_genesis

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))


print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

****** EJEMPLO v_semantico() ********
bow: [Synset('cut.v.24'), Synset('womanhood.n.02'), Synset('onion.n.01')]
syns1: [Synset('womanhood.n.02'), Synset('cut.v.24'), Synset('onion.n.01')]
syns2: [Synset('womanhood.n.02'), Synset('cut.v.24'), Synset('onion.n.01')]

SIMILITUD LIU - WANG
v_semantico(syns1, bow, medida=' lw ', info='  ' , ic=''): [0. 0. 0.]
v_semantico(syns2, bow, medida=' lw ', info='  ' , ic=''): [0. 0. 0.]

SIMILITUDES DE WN QUE NO REQUIEREN IC
v_semantico(syns1, bow, medida=' ps ', info='  ' , ic=''): [1. 1. 1.]
v_semantico(syns2, bow, medida=' ps ', info='  ' , ic=''): [1. 1. 1.]
v_semantico(syns1, bow, medida=' lch ', info='  ' , ic=''): [3.25809654 3.63758616 3.63758616]
v_semantico(syns2, bow, medida=' lch ', info='  ' , ic=''): [3.25809654 3.63758616 3.63758616]
v_semantico(syns1, bow, medida=' wup ', info='  ' , ic=''): [1. 1. 1.]
v_semantico(syns2, bow, medida=' wup ', info='  ' , ic=''): [1. 1. 1.]

SIMILITUDES DE WN QUE REQUIEREN IC. IC = BROWN
v_semantico(syn

In [15]:
# 8.2 Similitud de frase Liu-Wang
    # Entrada: Dos vectores semánticos
    # Salida: Similitud entre las dos frases, calculada como el coseno entre los dos vectores semánticos
    
def similitud_frase_lw(v1, v2):
        
    sim = coseno(v1, v2)
    
    return(sim)

# Ejemplo para comprobar funcionamiento similitud_frase_liu()

print('****** EJEMPLO similitud_frase() ********')   

v1 = v_semantico(syns1, bow, medida=medida, ic=ic, info=info)
v2 = v_semantico(syns2, bow, medida=medida, ic=ic, info=info)
print('Similitud a nivel de frase', similitud_frase_lw(v1, v2))
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

****** EJEMPLO similitud_frase() ********
Similitud a nivel de frase 1.0
****
TIEMPO TRANSCURRIDO: 75.87806558609009


In [16]:
# Cálculo y almacenamiento de la puntuación Liu-Wang a nivel de frase

if metodo_frase == 'lw':

    c['v1_ps'] = ''  # OK
    c['v2_ps'] = ''  # OK

    c['v1_lch'] = ''  # OK
    c['v2_lch'] = ''  # OK

    c['v1_wup'] = '' # OK
    c['v2_wup'] = '' # OK

    c['v1_res_brown'] = '' # Resultats no fiables quan la paraula no està al Corpus --> PROVAR IF ABANS; BoW_nv!!!!
    c['v2_res_brown'] = ''  # Resultats no fiables quan la paraula no està al Corpus --> PROVAR IF ABANS; BoW_nv!!!!

    c['v1_res_semcor'] = ''
    c['v2_res_semcor'] = ''

    c['v1_res_genesis'] = ''
    c['v2_res_genesis'] = ''

    c['v1_jcn_brown'] = '' # Components infinites quan les paraules són iguals --> ESTABLIR UN MÀXIM ; BoW_nv!!!!
    c['v2_jcn_brown'] = '' # Components infinites quan les paraules són iguals --> ESTABLIR UN MÀXIM ; BoW_nv!!!!

    c['v1_jcn_semcor'] = ''
    c['v2_jcn_semcor'] = ''

    c['v1_jcn_genesis'] = ''
    c['v2_jcn_genesis'] = ''

    c['v1_lin_brown'] = ''
    c['v2_lin_brown'] = ''

    c['v1_lin_semcor'] = ''
    c['v2_lin_semcor'] = ''

    c['v1_lin_genesis'] = ''
    c['v2_lin_genesis'] = ''
    

    c['puntuacion_lw_ps'] = 0.0  # OK, Pearson 0.66
    c['puntuacion_lw_lch'] = 0.0  # OK, Pearson 0.49
    c['puntuacion_lw_wup'] = 0.0 # OK, Pearson 0.42
    c['puntuacion_lw_res_brown'] = 0.0 # OK, Pearson 0.77 MILLORABLE 
    c['puntuacion_lw_res_genesis'] = 0.0
    c['puntuacion_lw_jcn_brown'] = 0.0  # Tots els resultats són zero.
    c['puntuacion_lw_jcn_semcor'] = 0.0
    c['puntuacion_lw_jcn_genesis'] = 0.0
    c['puntuacion_lw_lin_brown'] = 0.0
    c['puntuacion_lw_lin_semcor'] = 0.0
    c['puntuacion_lw_lin_genesis'] = 0.0

    # print_info = 's'

    for i in c.index:  
        gold_score = c.at[i, 'gold_score']
        sent1 = c.at[i, 'sent1']
        sent2 = c.at[i, 'sent2']
        syns1 = c.at[i, 'syns1']
        syns2 = c.at[i, 'syns2']
        if tipo_bow == 'nv':
            bow = c.at[i, 'bow_nv']
        else:
            bow = c.at[i, 'bow']
        start = time.time()
        if print_info == 's':
            print('FILA:', i)
            print('Gold score:', c.at[i, 'gold_score'])
            print('sent1:', c.at[i, 'sent1'])
            print('sent2:', c.at[i, 'sent2'])
            print('syns1:', c.at[i, 'syns1'])
            print('syns2:', c.at[i, 'syns2'])
            print('bow:', bow)

        try: c.at[i, 'v1_ps'] = v_semantico(c.at[i, 'syns1'], bow, medida='ps', ic='', info=info)
        except:  c.at[i, 'v1_ps'] = None
        try: c.at[i, 'v2_ps'] = v_semantico(c.at[i, 'syns2'], bow, medida='ps', ic='', info=info)
        except:  c.at[i, 'v2_ps'] = None
        try: c.at[i, 'puntuacion_lw_ps'] = similitud_frase_lw(c.at[i, 'v1_ps'], c.at[i, 'v2_ps'] )
        except:  c.at[i, 'puntuacion_lw_ps'] = None

        if print_info == 's':
            print('v1_ps', i, ': ', c.at[i, 'v1_ps'])
            print('v2_ps', i, ': ', c.at[i, 'v2_ps'])
            print('puntuacion_lw_ps', i, ': ', c.at[i, 'puntuacion_lw_ps'])

        try: c.at[i, 'v1_lch'] = v_semantico(c.at[i, 'syns1'], bow, medida='lch', ic='', info=info)
        except:  c.at[i, 'v1_lch'] = None
        try: c.at[i, 'v2_lch'] = v_semantico(c.at[i, 'syns2'], bow, medida='lch', ic='', info=info)
        except:  c.at[i, 'v2_lch'] = None
        try: c.at[i, 'puntuacion_lw_lch'] = similitud_frase_lw(c.at[i, 'v1_lch'], c.at[i, 'v2_lch'] )
        except:  c.at[i, 'puntuacion_lw_lch'] = None

        if print_info == 's':
            print('v1_lch', i, ': ', c.at[i, 'v1_lch'])
            print('v2_lch', i, ': ', c.at[i, 'v2_lch'])
            print('puntuacion_lw_lch', i, ': ', c.at[i, 'puntuacion_lw_lch'])

        try: c.at[i, 'v1_wup'] = v_semantico(c.at[i, 'syns1'], bow, medida='wup', ic='', info=info)
        except:  c.at[i, 'v1_wup'] = None
        try: c.at[i, 'v2_wup'] = v_semantico(c.at[i, 'syns2'], bow, medida='wup', ic='', info=info)
        except:  c.at[i, 'v2_wup'] = None
        try: c.at[i, 'puntuacion_lw_wup'] = similitud_frase_lw(c.at[i, 'v1_wup'], c.at[i, 'v2_wup'] )
        except:  c.at[i, 'puntuacion_lw_wup'] = None

        if print_info == 's':
            print('v1_wup', i, ': ', c.at[i, 'v1_wup'])
            print('v2_wup', i, ': ', c.at[i, 'v2_wup'])
            print('puntuacion_lw_wup', i, ': ', c.at[i, 'puntuacion_lw_wup'])

        ic = ic_brown

        try: c.at[i, 'v1_res_brown'] = v_semantico(c.at[i, 'syns1'], bow, medida='res', ic=ic_brown, info=info)
        except:  c.at[i, 'v1_res_brown'] = None
        try: c.at[i, 'v2_res_brown'] = v_semantico(c.at[i, 'syns2'], bow, medida='res', ic=ic_brown, info=info)
        except:  c.at[i, 'v2_res_brown'] = None
        try: c.at[i, 'puntuacion_lw_res_brown'] = similitud_frase_lw(c.at[i, 'v1_res_brown'], c.at[i, 'v2_res_brown'] )
        except:  c.at[i, 'puntuacion_lw_res_brown'] = None

        if print_info == 's':
            print('v1_res_brown', i, ': ', c.at[i, 'v1_res_brown'])
            print('v2_res_brown', i, ': ', c.at[i, 'v2_res_brown'])
            print('puntuacion_lw_res_brown', i, ': ', c.at[i, 'puntuacion_lw_res_brown'])      

        try: c.at[i, 'v1_jcn_brown'] = v_semantico(c.at[i, 'syns1'], bow, medida='jcn', ic=ic_brown, info=info)
        except:  c.at[i, 'v1_jcn_brown'] = None
        try: c.at[i, 'v2_jcn_brown'] = v_semantico(c.at[i, 'syns2'], bow, medida='jcn', ic=ic_brown, info=info)
        except:  c.at[i, 'v2_jcn_brown'] = None
        try: c.at[i, 'puntuacion_lw_jcn_brown'] = similitud_frase_lw(c.at[i, 'v1_jcn_brown'], c.at[i, 'v2_jcn_brown'] )
        except:  c.at[i, 'puntuacion_lw_jcn_brown'] = None

        if print_info == 's':
            print('v1_jcn_brown', i, ': ', c.at[i, 'v1_jcn_brown'])
            print('v2_jcn_brown', i, ': ', c.at[i, 'v2_jcn_brown'])
            print('puntuacion_lw_jcn_brown', i, ': ', c.at[i, 'puntuacion_lw_jcn_brown'])

        try: c.at[i, 'v1_lin_brown'] = v_semantico(c.at[i, 'syns1'], bow, medida='lin', ic=ic_brown, info=info)
        except:  c.at[i, 'v1_lin_brown'] = None
        try: c.at[i, 'v2_lin_brown'] = v_semantico(c.at[i, 'syns2'], bow, medida='lin', ic=ic_brown, info=info)
        except:  c.at[i, 'v2_lin_brown'] = None
        try: c.at[i, 'puntuacion_lw_lin_brown'] = similitud_frase_lw(c.at[i, 'v1_lin_brown'], c.at[i, 'v2_lin_brown'] )
        except:  c.at[i, 'puntuacion_lw_lin_brown'] = None

        if print_info == 's':
            print('v1_lin_brown', i, ': ', c.at[i, 'v1_lin_brown'])
            print('v2_lin_brown', i, ': ', c.at[i, 'v2_lin_brown'])
            print('puntuacion_lw_lin_brown', i, ': ', c.at[i, 'puntuacion_lw_lin_brown'])

        ic = ic_semcor

        try: c.at[i, 'v1_res_semcor'] = v_semantico(c.at[i, 'syns1'], bow, medida='res', ic=ic_semcor, info=info)
        except:  c.at[i, 'v1_res_semcor'] = None
        try: c.at[i, 'v2_res_semcor'] = v_semantico(c.at[i, 'syns2'], bow, medida='res', ic=ic_semcor, info=info)
        except:  c.at[i, 'v2_res_semcor'] = None
        try: c.at[i, 'puntuacion_lw_res_semcor'] = similitud_frase_lw(c.at[i, 'v1_res_semcor'], c.at[i, 'v2_res_semcor'])
        except:  c.at[i, 'puntuacion_lw_res_semcor'] = None

        if print_info == 's':
            print('v1_res_semcor', i, ': ', c.at[i, 'v1_res_semcor'])
            print('v2_res_semcor', i, ': ', c.at[i, 'v2_res_semcor'])
            print('puntuacion_lw_res_semcor', i, ': ', c.at[i, 'puntuacion_lw_res_semcor'])

        try: c.at[i, 'v1_jcn_semcor'] = v_semantico(c.at[i, 'syns1'], bow, medida='jcn', ic=ic_semcor, info=info)
        except:  c.at[i, 'v1_jcn_semcor'] = None
        try: c.at[i, 'v2_jcn_semcor'] = v_semantico(c.at[i, 'syns2'], bow, medida='jcn', ic=ic_semcor, info=info)
        except:  c.at[i, 'v2_jcn_semcor'] = None
        try: c.at[i, 'puntuacion_lw_jcn_semcor'] = similitud_frase_lw(c.at[i, 'v1_jcn_semcor'], c.at[i, 'v2_jcn_semcor'] )
        except:  c.at[i, 'puntuacion_lw_jcn_semcor'] = None

        if print_info == 's':
            print('v1_jcn_semcor', i, ': ', c.at[i, 'v1_jcn_semcor'])
            print('v2_jcn_semcor', i, ': ', c.at[i, 'v2_jcn_semcor'])
            print('puntuacion_lw_jcn_semcor', i, ': ', c.at[i, 'puntuacion_lw_jcn_semcor'])

        try: c.at[i, 'v1_lin_semcor'] = v_semantico(c.at[i, 'syns1'], bow, medida='lin', ic=ic_semcor, info=info)
        except:  c.at[i, 'v1_lin_semcor'] = None
        try: c.at[i, 'v2_lin_semcor'] = v_semantico(c.at[i, 'syns2'], bow, medida='lin', ic=ic_semcor, info=info)
        except:  c.at[i, 'v2_lin_semcor'] = None
        try: c.at[i, 'puntuacion_lw_lin_semcor'] = similitud_frase_lw(c.at[i, 'v1_lin_semcor'], c.at[i, 'v2_lin_semcor'] )
        except:  c.at[i, 'puntuacion_lw_lin_semcor'] = None

        if print_info == 's':
            print('v1_lin_semcor', i, ': ', c.at[i, 'v1_lin_semcor'])
            print('v2_lin_semcor', i, ': ', c.at[i, 'v2_lin_semcor'])
            print('puntuacion_lw_lin_semcor', i, ': ', c.at[i, 'puntuacion_lw_lin_semcor'])

        ic = ic_genesis

        try: c.at[i, 'v1_res_genesis'] = v_semantico(c.at[i, 'syns1'], bow, medida='res', ic=ic_genesis, info=info)
        except:  c.at[i, 'v1_res_genesis'] = None
        try: c.at[i, 'v2_res_genesis'] = v_semantico(c.at[i, 'syns2'], bow, medida='res', ic=ic_genesis, info=info)
        except:  c.at[i, 'v2_res_genesis'] = None
        try: c.at[i, 'puntuacion_lw_res_genesis'] = similitud_frase_lw(c.at[i, 'v1_res_genesis'], c.at[i, 'v2_res_genesis'] )
        except:  c.at[i, 'puntuacion_lw_res_genesis'] = None

        if print_info == 's':
            print('v1_res_genesis', i, ': ', c.at[i, 'v1_res_genesis'])
            print('v2_res_genesis', i, ': ', c.at[i, 'v2_res_genesis'])
            print('puntuacion_lw_res_genesis', i, ': ', c.at[i, 'puntuacion_lw_res_genesis'])

        try: c.at[i, 'v1_jcn_genesis'] = v_semantico(c.at[i, 'syns1'], bow, medida='jcn', ic=ic_genesis, info=info)
        except:  c.at[i, 'v1_jcn_genesis'] = None
        try: c.at[i, 'v2_jcn_genesis'] = v_semantico(c.at[i, 'syns2'], bow, medida='jcn', ic=ic_genesis, info=info)
        except:  c.at[i, 'v2_jcn_genesis'] = None
        try: c.at[i, 'puntuacion_lw_jcn_genesis'] = similitud_frase_lw(c.at[i, 'v1_jcn_genesis'], c.at[i, 'v2_jcn_genesis'] )
        except:  c.at[i, 'puntuacion_lw_jcn_genesis'] = None

        if print_info == 's':
            print('v1_jcn_genesis', i, ': ', c.at[i, 'v1_jcn_genesis'])
            print('v2_jcn_genesis', i, ': ', c.at[i, 'v2_jcn_genesis'])
            print('puntuacion_lw_jcn_genesis', i, ': ', c.at[i, 'puntuacion_lw_jcn_genesis'])

        try: c.at[i, 'v1_lin_genesis'] = v_semantico(c.at[i, 'syns1'], bow, medida='lin', ic=ic_genesis, info=info)
        except:  c.at[i, 'v1_lin_genesis'] = None
        try: c.at[i, 'v2_lin_genesis'] = v_semantico(c.at[i, 'syns2'], bow, medida='lin', ic=ic_genesis, info=info)
        except:  c.at[i, 'v2_lin_genesis'] = None
        try: c.at[i, 'puntuacion_lw_lin_genesis'] = similitud_frase_lw(c.at[i, 'v1_lin_genesis'], c.at[i, 'v2_lin_genesis'] )
        except:  c.at[i, 'puntuacion_lw_lin_genesis'] = None

        if print_info == 's':
            print('v1_lin_genesis', i, ': ', c.at[i, 'v1_lin_genesis'])
            print('v2_lin_genesis', i, ': ', c.at[i, 'v2_lin_genesis'])
            print('puntuacion_lw_lin_genesis', i, ': ', c.at[i, 'puntuacion_lw_lin_genesis'])    

        end = time.time()
        print(end-start)


    end_total = time.time()
    print('TIEMPO TRANSCURRIDO:', end_total-start_total)

0.02132725715637207
0.005952119827270508
0.013887882232666016
0.005455970764160156
0.006944417953491211
0.003968000411987305
0.005951404571533203
0.013392210006713867
0.021327495574951172
0.012896299362182617
0.01884770393371582
0.017360448837280273
0.008927345275878906
0.012896060943603516
0.010912895202636719
0.007935047149658203
0.005455970764160156
0.001984119415283203
0.008432865142822266
0.00942373275756836
0.009422540664672852
0.0069310665130615234
0.005456209182739258
0.0029752254486083984
0.004961252212524414
0.006446361541748047
0.003968000411987305
0.005953073501586914
0.012431144714355469
0.008896350860595703
0.009424448013305664
0.0044629573822021484
0.004462480545043945
0.00745844841003418
0.015346050262451172
0.005454063415527344
0.005456209182739258
0.008432149887084961
0.006945610046386719
0.004462242126464844
0.0024797916412353516
0.003968000411987305
0.011904001235961914
0.005953073501586914
0.009918928146362305
0.00793600082397461
0.0077762603759765625
0.00746822357

0.011376380920410156
0.007440328598022461
0.012430191040039062
0.011377334594726562
0.01044607162475586
0.013856649398803711
0.014417648315429688
0.009424209594726562
0.005454540252685547
0.01292729377746582
0.019515037536621094
0.007437705993652344
0.008432865142822266
0.011406898498535156
0.01292562484741211
0.011374473571777344
0.028270483016967773
0.013889551162719727
0.01788640022277832
0.006453275680541992
0.009241819381713867
0.009528160095214844
0.009917497634887695
0.013393402099609375
0.013390302658081055
0.005455970764160156
0.008962392807006836
0.00492548942565918
0.008432149887084961
0.010912179946899414
0.011408329010009766
0.006942033767700195
0.014386177062988281
0.012399911880493164
0.007440328598022461
0.010417699813842773
0.013886451721191406
0.014877796173095703
0.014885902404785156
0.012898445129394531
0.008430719375610352
0.012926101684570312
0.014899015426635742
0.009920358657836914
0.008435249328613281
0.007436513900756836
0.00796365737915039
0.00597643852233886

0.005425691604614258
0.015401601791381836
0.023345232009887695
0.010877609252929688
0.007445573806762695
0.008432865142822266
0.007439613342285156
0.011407852172851562
0.012894630432128906
0.012927532196044922
0.011904001235961914
0.007439851760864258
0.0034706592559814453
0.0064465999603271484
0.007443666458129883
0.007930517196655273
0.00793600082397461
0.005463361740112305
0.01139974594116211
0.011879205703735352
0.020334720611572266
0.009424209594726562
0.01091146469116211
0.009920597076416016
0.009918212890625
0.007439136505126953
0.008432149887084961
0.01636791229248047
0.028272390365600586
0.024303674697875977
0.0064487457275390625
0.013887405395507812
0.005455493927001953
0.003645658493041992
0.008431196212768555
0.020114421844482422
0.008429527282714844
0.01931452751159668
0.00647735595703125
0.006419181823730469
0.04166579246520996
0.0054547786712646484
0.006449222564697266
0.0054547786712646484
0.007440328598022461
0.005455732345581055
0.005456209182739258
0.0059833526611328

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:9: RuntimeWarning: invalid value encountered in double_scalars
  if __name__ == '__main__':


0.027776718139648438
0.02132701873779297
0.023809432983398438
0.013362884521484375
0.0035016536712646484
0.005426168441772461
0.006446123123168945
0.015375852584838867
0.00794529914855957
0.0128936767578125
0.005920886993408203
0.008430719375610352
0.011407852172851562
0.008936166763305664
0.007952451705932617
0.007773637771606445
0.007966995239257812
0.0074100494384765625
0.010441303253173828
0.0163726806640625
0.015840530395507812
0.005455732345581055
0.008961200714111328
0.009498357772827148
0.0053751468658447266
0.010414838790893555
0.008930206298828125
0.00893092155456543
0.01336216926574707
0.012918472290039062
0.006474018096923828
0.011912822723388672
0.005918264389038086
0.009423494338989258
0.028798818588256836
0.005490303039550781
0.010877847671508789
0.018383026123046875
0.014847993850708008
0.008956193923950195
0.008403301239013672
0.001984119415283203
0.025824308395385742
0.005951642990112305
0.004929542541503906
0.010440826416015625
0.01039886474609375
0.01785492897033691

...

0.015903711318969727
0.022288799285888672
0.0243380069732666
0.021300792694091797
0.009455442428588867
0.015374422073364258
0.008908271789550781
0.010942697525024414
0.01388692855834961
0.013888120651245117
0.0074920654296875
0.00837397575378418
0.012895345687866211
0.005455970764160156
0.010909557342529297
0.0218050479888916
0.022351980209350586
0.011406660079956055
0.004960298538208008
0.0054547786712646484
0.026289939880371094
0.02188253402709961
0.0059490203857421875
0.004467487335205078
0.01239919662475586
0.00991964340209961
0.003968000411987305
0.009920358657836914
0.003967761993408203
0.013391733169555664
0.013387680053710938
0.017385005950927734
0.02330493927001953
0.0049588680267333984
0.015902996063232422
0.004462242126464844
0.004846334457397461
0.011910200119018555
0.0009777545928955078
0.014910459518432617
0.02129840850830078
0.021822690963745117
0.006446361541748047
0.01686382293701172
0.01438450813293457
0.014879226684570312
0.00793600082397461
0.006447792053222656
0.00

0.005455732345581055
0.010913372039794922
0.02083301544189453
0.05356788635253906
0.05704188346862793
0.025790929794311523
0.014880895614624023
0.009422540664672852
0.006943702697753906
0.013392210006713867
0.011904239654541016
0.008928060531616211
0.013919591903686523
0.005951881408691406
0.029265165328979492
0.008926630020141602
0.010416507720947266
0.005422830581665039
0.028765439987182617
0.011409282684326172
0.012926340103149414
0.009421348571777344
0.02480030059814453
0.008893251419067383
0.01983928680419922
0.008961200714111328
0.0074367523193359375
0.020832061767578125
0.004457235336303711
0.01091146469116211
0.011409282684326172
0.01683974266052246
0.009955406188964844
0.010442018508911133
0.009426593780517578
0.008896350860595703
0.02529740333557129
0.03227114677429199
0.020801782608032227
0.011901617050170898
0.008431196212768555
0.030751466751098633
0.012400388717651367
0.010417699813842773
0.007439851760864258
0.013888359069824219
0.047617197036743164
0.014382123947143555


0.01485443115234375
0.011407136917114258
0.016864299774169922
0.010942220687866211
0.004463911056518555
0.012369871139526367
0.011438846588134766
0.013857364654541016
0.021854639053344727
0.020303964614868164
0.01785588264465332
0.019342660903930664
0.007936954498291016
0.005952119827270508
0.01835179328918457
0.007440805435180664
0.006448507308959961
0.023807048797607422
0.017858028411865234
0.00991964340209961
0.019840240478515625
0.024303913116455078
0.023314714431762695
0.007935047149658203
0.0054585933685302734
0.037230491638183594
0.01735830307006836
0.008432626724243164
0.023839473724365234
0.009888172149658203
0.016398191452026367
0.010446786880493164
0.01441192626953125
0.02083563804626465
0.013358592987060547
0.008931398391723633
0.034720420837402344
0.01140737533569336
0.014879941940307617
0.0074388980865478516
0.018352270126342773
0.029790163040161133
0.009424924850463867
0.004992485046386719
0.006911039352416992
0.017857789993286133
0.015374183654785156
0.02283596992492675

0.03274226188659668
0.020831584930419922
0.001983165740966797
0.004460811614990234
0.006447553634643555
0.007935762405395508
0.01736164093017578
0.007494449615478516
0.010417699813842773
0.015405893325805664
0.012372255325317383
0.011437416076660156
0.01137685775756836
0.011936426162719727
0.006944179534912109
0.013918399810791016
0.008894920349121094
0.007440328598022461
0.010415792465209961
0.008431673049926758
0.014889240264892578
0.02480173110961914
0.04215884208679199
0.008463144302368164
0.007947206497192383
0.0039517879486083984
0.00991964340209961
0.013887882232666016
0.027776002883911133
0.02281665802001953
0.029263734817504883
0.022325754165649414
0.007441282272338867
0.024304866790771484
0.010910511016845703
0.008927345275878906
0.005455493927001953
0.014877557754516602
0.013889551162719727
0.02579212188720703
0.012895345687866211
0.02480030059814453
0.006445169448852539
0.012433528900146484
0.006413936614990234
0.029761314392089844
0.006942272186279297
0.014383792877197266


0.013390064239501953
0.03227639198303223
0.03418898582458496
0.014414548873901367
0.013884544372558594
0.009916543960571289
0.016890287399291992
0.027775287628173828
0.012925863265991211
0.013396739959716797
0.05108785629272461
0.013390541076660156
0.019840478897094727
0.015868663787841797
TIEMPO TRANSCURRIDO: 171.47885131835938


In [17]:
x = datetime.datetime.now()
c.to_excel(x.strftime("%y%m%d-%H%M-") + criterios + '.xlsx')

end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

TIEMPO TRANSCURRIDO: 199.11921644210815
